# LLM API 标注 — 三模型对比（MiMo / Deepseek / Qwen）

**流程**：读取 `Data/LLM.csv` 的评论 → 分别调用三个模型 API → 判定「负面性」「侮辱性」→ 结果填入对应模型的列 → 写入**新文件**。

## 重要约束

- **提示词四模块拼装**：① 系统 / ② 格式 / ③ 标签定义 / ④ 额外；`engineered` 四块全用，`zero_shot` 只用前三块，唯一自变量是 ④「额外提示词」
- **两种提示词模式分文件输出**：`engineered` → `Data/LLM/LLM_prompt_predictions.csv`，`zero_shot` → `Data/LLM/LLM_predictions.csv`
- **统一用 OpenAI 官方 SDK** 发请求（`client.chat.completions.create`），三家模型只是 `base_url` / `model` 不同
- 代码末尾有**文件指纹校验**，运行前后对比 MD5，证明源文件未被改动
- **断点续跑**：420 条 × 3 模型 = 1260 次调用，中途报错/中断不会白跑，已完成的响应存在 `Data/llm_raw/` 下，重跑自动跳过

## 使用步骤

1. 在「配置」cell 填入各模型的 `api_key`；`base_url` 留空即 OpenAI 官方端点，走第三方/中转时填其 OpenAI 兼容地址
2. 按顺序运行「读数据」→「提示词」→「单条调用测试」→「批量标注」→「导出 CSV」
3. 先用测 1 条确认 API 通、格式对，再跑全量
4. **`model` 名必须与端点实际支持的名称逐字一致（大小写敏感）**。名字写错时，单条测试会打印 `[致命] ... The supported API model names are ...`，把报错里列出的名字填回配置 cell 即可（例：DeepSeek 端点可用 `deepseek-v4-pro` / `deepseek-flash`）

In [ ]:
import hashlib
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from openai import OpenAI

DATA_DIR = Path("../Data/LLM")

# 本次运行的模型（改这一行切换）
MODEL_KEY = "MiMo"       # 可选: "Deepseek" | "Qwen" | "MiMo"
PROMPT_MODE = "zero_shot"   # 可选: "engineered"（提示词工程）| "zero_shot"（零样本）

# 路径
IN_PATH = DATA_DIR / "LLM.csv"

# 两种提示词模式
OUT_FILES = {
    "engineered": DATA_DIR / "LLM_prompt_predictions.csv",   # 提示词工程输出文件
    "zero_shot":  DATA_DIR / "LLM_predictions.csv",          # 零样本输出文件
}
assert PROMPT_MODE in OUT_FILES, f"PROMPT_MODE 只能是 {list(OUT_FILES)}"
OUT_PATH = OUT_FILES[PROMPT_MODE]                  # 结果输出

RAW_DIR = DATA_DIR / "llm_raw"                     # 原始响应存档 + 断点续跑
if not RAW_DIR.exists():
    RAW_DIR.mkdir(parents=True)

# 三个模型的 API 配置
MODELS = {
    "MiMo": {
        "base_url": "https://api.xiaomimimo.com/v1",
        "api_key": "",
        "model": "mimo-v2.6-pro",   # 端点实际可用名：mimo-v2.6-pro / mimo-v2.6-flash / mimo-v2.6-pro-ultraspeed
    },
    "Deepseek": {
        "base_url": "https://api.deepseek.com",
        "api_key": "",
        "model": "deepseek-v4-pro",   # 端点实际可用名：deepseek-v4-pro / deepseek-flash
    },
    "Qwen": {
        "base_url": "https://llm-dowo0a0yqphez52i.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
        "api_key": "",
        "model": "qwen3.8-max",   # 端点实际可用名：qwen3.8-max
    },
}

# 调用参数
TEMPERATURE = 0.0        # 分类任务用 0 保证可复现
MAX_TOKENS = 4096        # 输出 token 上限（足够 JSON 输出）
TIMEOUT = 60             # 单次请求超时（秒）
MAX_RETRY = 3            # 失败重试次数
MAX_WORKERS = 4          # 并发线程数（注意 API 速率限制）
SLEEP_BETWEEN = 0.3      # 每次请求后间隔（秒），防限流

RESPONSE_FORMAT_JSON = True   # 用 OpenAI 的 JSON 模式强制结构化输出
TOKEN_PARAM = "max_tokens"    # 另一种取值 "max_completion_tokens"（新版 OpenAI 模型要求）
# 以上两项端点若不兼容会自动降级/切换，不必手改，跑完看控制台的 [降级] / [自适应] 提示即可

# 配置自检
_cfg = MODELS[MODEL_KEY]
_ready = bool(_cfg["api_key"])
print()
print(f"本次运行的模型: {MODEL_KEY}（改配置 cell 顶部的 MODEL_KEY 切换）")
print(f"模型名:         {_cfg['model']}")
print(f"端点:           {_cfg['base_url']}")
print(f"提示词模式:     {PROMPT_MODE}")
print(f"JSON 模式:      {'开' if RESPONSE_FORMAT_JSON else '关'}")
print(f"输出文件:       {OUT_PATH}")
print(f"断点文件:       {RAW_DIR / f'{MODEL_KEY}_{PROMPT_MODE}_responses.jsonl'}")
print(f"API 状态:       {'已配置' if _ready else '未配置 —— 请填入 ' + MODEL_KEY + ' 的 api_key'}")
print()
print("两种模式各自归档:")
for mode, path in OUT_FILES.items():
    flag = " ← 本次写入" if mode == PROMPT_MODE else ""
    print(f"  {mode:10} → {path.name}{flag}")



本次运行的模型: MiMo（改配置 cell 顶部的 MODEL_KEY 切换）
模型名:         mimo-v2.6-pro
端点:           https://api.xiaomimimo.com/v1
提示词模式:     zero_shot
JSON 模式:      开
输出文件:       ../Data/LLM/LLM_predictions.csv
断点文件:       ../Data/LLM/llm_raw/MiMo_zero_shot_responses.jsonl
API 状态:       已配置

两种模式各自归档:
  engineered → LLM_prompt_predictions.csv
  zero_shot  → LLM_predictions.csv ← 本次写入


## 读取数据

读取评论与人工标注标签。源文件在此之后会被计算指纹，用于最终校验未被改动。

In [2]:
# 源文件指纹（末尾用于校验文件未被修改）
IN_MD5 = hashlib.md5(IN_PATH.read_bytes()).hexdigest()

df_in = pd.read_csv(IN_PATH)
LABEL_COLS = [c for c in df_in.columns if c not in ("类型", "评论")]
print(f"读取 {IN_PATH}：{df_in.shape[0]} 行")
print(f"列名: {list(df_in.columns)}")
print(f"\n示例评论：")
for i in range(3):
    print(f"  [{df_in.loc[i, '类型']}] {df_in.loc[i, '评论'][:40]}")

读取 ../Data/LLM/LLM.csv：420 行
列名: ['类型', '评论', '负面性MiMo', '侮辱性MiMo', '负面性Deepseek', '侮辱性Deepseek', '负面性Qwen', '侮辱性Qwen']

示例评论：
  [语气型差评] 用料扎实到咬不动 牙医听了都连夜谢谢你们家🤣
  [语气型差评] 菜里面竟然附赠了一条🐛蠕动的赠品，真是太贴心了😭
  [语气型差评] 老板仁义啊！居然拿半个世纪前的古董来做菜！菜叶枯黄得像秋天的落叶，不得不分五期给


## 提示词设计

提示词由**四个可独立拼装的模块**构成，两种模式的区别只在最后一块：

| 模块 | 作用 | 字符数 | zero_shot | engineered |
| --- | --- | --- | --- | --- |
| ① 系统提示词 | `system` 角色，定身份与输出格式 | 84 | 有 | 有 |
| ② 格式提示词 | 一条示例，演示 JSON 输出格式 | 48 | 有 | 有 |
| ③ 标签定义提示词 | 两个标签的判定口径 | 186 | 有 | 有 |
| ④ 额外提示词 | 规避手段清单 + 层级约束 | 301 | — | 有 |

```
零样本    zero_shot  = ① + ② + ③        257 字符
提示词工程 engineered = ① + ② + ③ + ④    559 字符
```

**核心难点**（从前面的 ML 实验已知）：侮辱性不是词汇级现象，而是**语用级**现象——
- 反讽/阴阳怪气（「五星好评了捏」）没有半个脏字，但确是负面+侮辱
- 骂人被**刻意混淆**：谐音（沙币=傻逼）、拆字（歹匕=死）、形近字（出生=畜生）、拼音缩写（nmsl）、英文变体（S0n of b1tch）

**消融设计**：②③ 两臂共用，唯一自变量是 ④「额外提示词」——即"**把规避手段与层级约束显式告知模型，是否提升性能**"。② 格式提示词放在两臂而非只给 engineered，一是演示输出格式，二是避免"只有一臂带示例"引入的单边偏置（稀有类上尤其致命）。

**措辞提醒**：本项目的 `zero_shot` 严格说是「**无额外规则告知**」，不是字面意义的零样本——它仍含标签定义与一条格式示例。报告里按这个措辞描述，别直接写 "zero-shot prompting"。


In [3]:
# 提示词由四个可独立拼装的模块构成
#   ① 系统提示词  ② 格式提示词  ③ 标签定义提示词  ④ 额外提示词
# 两种模式的唯一差别：engineered 多一个「额外提示词」模块
#   engineered = ① + ② + ③ + ④
#   zero_shot  = ① + ② + ③

# ---------- ① 系统提示词（走 system 角色，两模式相同） ----------
SYS_PROMPT = (
    "你是中文外卖评论文本标注专家。请判断给定评论在两个方面是否成立，只输出 JSON，不要任何多余文字。\n"
    "输出格式严格为：{\"负面性\": 0或1, \"侮辱性\": 0或1}\n"
)

# ---------- ② 格式提示词（示例演示输出格式；两模式相同） ----------
# 注意：块内的字面量花括号必须写成 {{ }}，否则末尾的 .format(review=...) 会抛 KeyError
FORMAT_PROMPT = """
示例
评论：这家店铺的外卖真的非常好吃！
输出：{{"负面性": 0, "侮辱性": 0}}
"""

# ---------- ③ 标签定义提示词（两模式相同） ----------
DEFINITIONS = """【负面性】=1：顾客对本次消费体验给出负面评价或表达不满。包括：对口味或服务的直接抱怨（难吃、太少、太咸）、反讽阴阳怪气（表面夸赞实则贬损）、对餐品/配送/服务的明确否定。对于褒义评价，则判 0。
【侮辱性】=1：评论中出现对商家、骑手、服务员等具体对象的辱骂、人身攻击、人格贬损、诅咒。仅吐槽餐品或表达不满、但未针对人的，判 0（此时负面性仍为 1）。"""

# ---------- ④ 额外提示词（只有 engineered 模式有） ----------
EXTRA_PROMPT = """注意本语料大量使用规避手段，根据谐音转码后的结果判定负面性和侮辱性：
· 谐音/近音字（例如：出生=畜生、沙币=傻逼、泥马=你妈、特木的=他妈的、乌鸡鲅鱼=无鸡巴语、虾扯蛋=瞎扯淡、饭溅=犯贱、食不食油饼=是不是有病、西内=死ね（日语））
· 拆字/形近字（歹匕=死、女马=妈、亻尔=你、米共=粪、犭句=狗、爪巴=爬）
· 拼音缩写与英文变体（nmsl、wqnmlgb、S0n of b1tch、$h1t、A55h0le、funny mud go pee，其中常见字母变体：0=O, 1=I, 3=E, 4=A, 5=S, $=S）
【层级约束】侮辱性=1 时，负面性必须=1（骂商家一定是在给差评）。"""

# ---------- 两种模式的拼装（只差 EXTRA_PROMPT） ----------
# 拼装顺序按 spec：格式提示词 → 标签定义提示词 →（额外提示词）；改顺序只需调整下面两行
BLOCKS = {
    "zero_shot":  [FORMAT_PROMPT, DEFINITIONS],
    "engineered": [FORMAT_PROMPT, DEFINITIONS, EXTRA_PROMPT],
}
assert PROMPT_MODE in BLOCKS, f"PROMPT_MODE 只能是 {list(BLOCKS)}"

TAIL = "\n\n评论：{review}\n输出 JSON："      # 评论槽位，两模式完全相同，不影响对照
PROMPT_TEMPLATE = "\n".join(BLOCKS[PROMPT_MODE]) + TAIL

# ---------- 预览：拼装结构 + 两模式差异 ----------
_other = "engineered" if PROMPT_MODE == "zero_shot" else "zero_shot"
_tpl_oth = "\n".join(BLOCKS[_other]) + TAIL
_eng_total = len("\n".join(BLOCKS["engineered"]) + TAIL)
_zs_total = len("\n".join(BLOCKS["zero_shot"]) + TAIL)

print()
print("=" * 68)
print(f"当前模式: {PROMPT_MODE}    用户提示词拼装结构:")
print("=" * 68)
_lbl = ["格式提示词", "标签定义提示词", "额外提示词"]
for _i, _blk in enumerate(BLOCKS[PROMPT_MODE]):
    print(f"  [{_i + 1}] {_lbl[_i]:8} {len(_blk):4} 字符")
print(f"  [尾] {'评论槽位':8} {len(TAIL):4} 字符（两模式相同）")
print(f"  合计 {len(PROMPT_TEMPLATE)} 字符")

print()
print("两模式对照:")
print(f"  zero_shot  {_zs_total:4} 字符（① + ② + ③ + 尾）")
print(f"  engineered {_eng_total:4} 字符（① + ② + ③ + ④ + 尾）")
_delta = len(EXTRA_PROMPT) + 1        # 额外提示词 + 一个分隔换行
print(f"  差异 = 「额外提示词」{len(EXTRA_PROMPT)} 字符 + 1 个换行 = {_delta} 字符"
      f"，占 engineered 的 {_delta / _eng_total:.1%}")



当前模式: zero_shot    用户提示词拼装结构:
  [1] 格式提示词      48 字符
  [2] 标签定义提示词   177 字符
  [尾] 评论槽位       22 字符（两模式相同）
  合计 248 字符

两模式对照:
  zero_shot   248 字符（① + ② + ③ + 尾）
  engineered  550 字符（① + ② + ③ + ④ + 尾）
  差异 = 「额外提示词」301 字符 + 1 个换行 = 302 字符，占 engineered 的 54.9%


## 单条调用与解析

- 用 **OpenAI 官方 SDK**发起请求：`OpenAI(api_key=..., base_url=...)` → `client.chat.completions.create(...)`。`base_url` 留空即 OpenAI 官方端点，填第三方地址即走其 OpenAI 兼容接口（MiMo / Deepseek / Qwen 兼容模式均适用）
- **JSON 模式**：`response_format={"type": "json_object"}` 强制结构化输出（`RESPONSE_FORMAT_JSON` 开关）。端点不支持时自动降级为普通模式，只打印一行 `[降级]`，不中断批量任务
- **token 参数自适应**：新版 OpenAI 模型要求 `max_completion_tokens` 而非 `max_tokens`；端点报错时自动切换并记住（`[自适应]`），同样不必手改
- 解析返回的 JSON，容错处理（模型可能裹 ```json 代码块或加解释文字）
- 失败重试，最终失败返回 `None`（不写 0，避免把「调用失败」误当成「判定为0」）
- 若 `finish_reason == "length"`（输出被截断），错误日志会单独标出，便于区分「模型话太多」与「接口报错」
- 客户端按模型缓存复用（`_CLIENTS`），并发下不会每次请求都重建连接
- **配置类错误立即熔断**：模型名写错 / 密钥无效 / 权限不足这类错误重试无用。`call_model` 会立刻放弃该条并记入 `LAST_FATAL`（不再原地重试 3 次）；批量层若发现「一条都没成功、且已出现配置类致命错误」，直接中止剩余任务，不会拿 420 条去撞端点。控制台打印 `[致命]` / `[熔断]` 并给出修正指引


In [4]:
import re


def parse_reply(text: str):
    """从模型回复中提取 (负面性, 侮辱性)；解析失败返回 None。"""
    if not text:
        return None
    # 去掉 ```json ... ``` 包裹
    cleaned = re.sub(r"```(?:json)?|```", "", text).strip()
    # 先整体解析（JSON 模式下通常就是纯 JSON）；失败再取第一个扁平对象（非贪婪）
    cands = [cleaned]
    m = re.search(r"\{[^{}]*\}", cleaned, re.DOTALL)
    if m:
        cands.append(m.group(0))
    obj = None
    for c in cands:
        try:
            obj = json.loads(c)
            break
        except json.JSONDecodeError:
            obj = None
    if not isinstance(obj, dict):
        return None
    try:
        neg, ins = int(obj.get("负面性")), int(obj.get("侮辱性"))
    except (TypeError, ValueError):
        return None
    if neg not in (0, 1) or ins not in (0, 1):
        return None
    if ins == 1 and neg == 0:          # 强制层级约束：侮辱 ⇒ 负面
        neg = 1
    return neg, ins


_CLIENTS = {}       # {model_key: OpenAI}，按模型缓存客户端
_PREFS = {}         # {model_key: {"json": bool, "token_key": str}}，端点兼容性自适应
ADAPT_LOG = []      # 自适应记录，跑完可复核用
LAST_FATAL = {}     # {model_key: 错误信息}，只记「重试也没用」的配置类错误，供批量层熔断


def get_client(model_key: str) -> OpenAI:
    """按模型配置构造并缓存 OpenAI 客户端；base_url 留空 = 官方端点。"""
    if model_key not in _CLIENTS:
        cfg = MODELS[model_key]
        if not cfg["api_key"]:
            raise ValueError(f"模型 {model_key} 的 API 未配置（api_key 为空）")
        _CLIENTS[model_key] = OpenAI(
            api_key=cfg["api_key"],
            base_url=cfg["base_url"] or None,     # None → 官方 https://api.openai.com/v1
            timeout=TIMEOUT,
            max_retries=0,                        # 重试统一由 call_model 控制，避免两套退避叠加
        )
    return _CLIENTS[model_key]


def _pref(model_key: str) -> dict:
    """该模型当前生效的请求偏好（端点不支持时会被自动改写）。"""
    return _PREFS.setdefault(model_key, {"json": RESPONSE_FORMAT_JSON, "token_key": TOKEN_PARAM})


def call_model(model_key: str, review: str, retries: int = MAX_RETRY):
    """调用单个模型（OpenAI 兼容接口），返回 (负面性, 侮辱性) 或 None。"""
    cfg = MODELS[model_key]
    client = get_client(model_key)

    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": PROMPT_TEMPLATE.format(review=review)},
    ]

    last_err = None
    for attempt in range(1, retries + 1):
        p = _pref(model_key)
        kwargs = {
            "model": cfg["model"],
            "messages": messages,
            "temperature": TEMPERATURE,
            p["token_key"]: MAX_TOKENS,
        }
        if p["json"]:
            kwargs["response_format"] = {"type": "json_object"}

        try:
            resp = client.chat.completions.create(**kwargs)
        except Exception as e:
            emsg = f"{type(e).__name__}: {e}"
            low = emsg.lower()
            # 兼容端点常见两种参数不兼容：自动降级并记住，避免 423 条全废
            if p["json"] and "response_format" in low:
                p["json"] = False
                ADAPT_LOG.append((model_key, "关闭 response_format（端点不支持 JSON 模式）"))
                print(f"  [降级] {model_key}: 端点不支持 response_format，改用普通模式")
                last_err = emsg
                continue
            if "max_tokens" in low and any(k in low for k in
                                           ("unsupported", "unrecognized", "not supported", "unknown")):
                p["token_key"] = ("max_completion_tokens"
                                  if p["token_key"] == "max_tokens" else "max_tokens")
                ADAPT_LOG.append((model_key, f"改用 {p['token_key']}"))
                print(f"  [自适应] {model_key}: 端点不接受上一个 token 参数，改用 {p['token_key']}")
                last_err = emsg
                continue
            # 永久性错误（模型名写错 / 密钥无效 / 权限不足 / 请求非法）重试无意义：
            # 立即放弃该条并留标记，免得 423 条 × MAX_RETRY 次无效请求 + 退避空等
            if (type(e).__name__ in ("AuthenticationError", "PermissionDeniedError",
                                     "NotFoundError", "BadRequestError")
                    or "invalid_request_error" in low):
                LAST_FATAL[model_key] = emsg
                print(f"  [致命] {model_key} | {review[:20]}... | {emsg[:300]}")
                return None
            last_err = emsg
            if attempt < retries:
                time.sleep(2 * attempt)       # 指数退避
            continue

        if not getattr(resp, "choices", None):
            last_err = "响应中没有 choices"
            if attempt < retries:
                time.sleep(1)
            continue

        choice = resp.choices[0]
        content = choice.message.content or ""
        parsed = parse_reply(content)
        if parsed is not None:
            return parsed

        _hint = "（输出被截断，finish_reason=length）" if choice.finish_reason == "length" else ""
        last_err = f"解析失败{_hint}: {content[:200]}"
        if attempt < retries:                 # temperature=0 下重试结果通常一致，仅再试一次
            time.sleep(1)

    print(f"  [失败] {model_key} | {review[:20]}... | {last_err}")
    return None


print("调用函数已定义（OpenAI SDK，未执行）")


调用函数已定义（OpenAI SDK，未执行）


### 先用单条测通再跑全量

In [5]:
# 测试一条（填好 API 后运行此 cell 验证连通性）
TEST_INDEX = 0

try:
    result = call_model(MODEL_KEY, df_in.loc[TEST_INDEX, "评论"])
    print(f"模型: {MODEL_KEY}")
    print(f"评论: {df_in.loc[TEST_INDEX, '评论']}")
    if result is None:
        # 注意：失败时返回 None，不能直接 result[0]——那样会抛 TypeError 把真正的错误盖掉
        print("结果: [失败] 未取得有效结果")
        if LAST_FATAL.get(MODEL_KEY):
            print("\n致命错误（配置问题，重试无用）:")
            print(f"  {LAST_FATAL[MODEL_KEY]}")
            print(f"\n→ 请核对配置 cell 里 MODELS['{MODEL_KEY}'] 的 model / base_url / api_key")
        else:
            print("→ 失败原因见上面 [失败] 行；若是超时或限流，直接重跑本 cell 即可")
    else:
        print(f"结果: 负面性={result[0]}, 侮辱性={result[1]}")
        print("\n连通性正常，可以运行下面的「执行标注」cell 跑全量。")
except ValueError as e:
    print(f"[跳过] {e}")
    print("→ 请先在配置 cell 填入 api_key（base_url 留空即走 OpenAI 官方端点）")


模型: MiMo
评论: 用料扎实到咬不动 牙医听了都连夜谢谢你们家🤣
结果: 负面性=1, 侮辱性=0

连通性正常，可以运行下面的「执行标注」cell 跑全量。


## 批量标注（并发 + 断点续跑）

420 条 × 3 模型 = 1260 次请求。设计：

- **并发**：`ThreadPoolExecutor`，默认 4 线程（注意 API QPS 限制，可调小）
- **断点续跑**：每个模型的响应实时落盘到 `Data/llm_raw/{模型}_responses.jsonl`，重跑时先读已完成的索引并跳过，中断不白跑
- **原始响应存档**：连同 review、用时一并存下，便于事后复核与报告附录

In [6]:
def raw_path(model_key: str) -> Path:
    # 文件名带上提示词模式，避免 engineered / zero_shot 两次运行的断点互相污染
    return RAW_DIR / f"{model_key}_{PROMPT_MODE}_responses.jsonl"


def load_done(model_key: str) -> dict:
    """读取已完成的响应（断点续跑）。"""
    p = raw_path(model_key)
    done = {}
    if p.exists():
        with open(p, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("负面性") is not None:
                    done[rec["idx"]] = rec
    return done


def annotate_model(model_key: str) -> dict:
    """对全部评论调用某模型，返回 {idx: 结果}。支持断点续跑。"""
    if not MODELS[model_key]["api_key"]:
        print(f"[跳过] {model_key}: API 未配置（api_key 为空）")
        return {}

    LAST_FATAL.pop(model_key, None)     # 清掉历史致命标记，避免陈旧状态误触发熔断
    done = load_done(model_key)
    todo = [(i, r) for i, r in enumerate(df_in["评论"]) if i not in done]
    print(f"{model_key}: 已完成 {len(done)} 条，待处理 {len(todo)} 条")

    if todo:
        with open(raw_path(model_key), "a", encoding="utf-8") as f, \
             ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(call_model, model_key, review): (i, review) for i, review in todo}
            n_ok = 0
            n_success = 0
            aborted = False
            for fut in as_completed(futures):
                i, review = futures[fut]
                try:
                    res = fut.result()
                except Exception as e:
                    res = None
                    print(f"  [异常] idx={i}: {e}")
                rec = {"idx": i, "评论": review,
                       "负面性": res[0] if res else None,
                       "侮辱性": res[1] if res else None}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()                       # 实时落盘，断电也不丢
                done[i] = rec
                n_ok += 1
                n_success += res is not None
                if n_ok % 20 == 0:
                    print(f"  ... {n_ok}/{len(todo)}")
                # 熔断：一条都没成功，且已出现配置类致命错误 → 立刻中止，别把剩下的全撞一遍
                if n_success == 0 and LAST_FATAL.get(model_key):
                    print(f"\n[熔断] 前 {n_ok} 条全部失败，判定为配置错误，中止剩余 {len(todo) - n_ok} 条")
                    print(f"       {LAST_FATAL[model_key][:200]}")
                    print("       → 修正配置 cell 后重跑本 cell；失败记录不会被当作已完成，会自动重试")
                    pool.shutdown(wait=False, cancel_futures=True)
                    aborted = True
                    break
                time.sleep(SLEEP_BETWEEN)
        print(f"{model_key}: 本轮完成 {n_ok} 条" + ("（已熔断中止）" if aborted else ""))

    ok = sum(1 for r in done.values() if r.get("负面性") is not None)
    print(f"{model_key}: 合计有效 {ok}/{len(df_in)}\n")
    return done


print("批量函数已定义（未执行）")

批量函数已定义（未执行）


In [7]:
# ===== 执行标注：只跑本次 MODEL_KEY 指定的模型（需要 API 配置）=====
results = {}
results[MODEL_KEY] = annotate_model(MODEL_KEY)

_n_valid = sum(1 for r in results[MODEL_KEY].values() if r.get("负面性") is not None)
print(f"模型 {MODEL_KEY} 标注完成，有效条数: {_n_valid}/{len(df_in)}")
print("→ 换模型请修改配置 cell 的 MODEL_KEY，然后重跑「执行标注」+「导出结果 CSV」两个 cell")


MiMo: 已完成 0 条，待处理 420 条
  ... 20/420
  ... 40/420
  ... 60/420
  ... 80/420
  ... 100/420
  ... 120/420
  ... 140/420
  ... 160/420
  ... 180/420
  ... 200/420
  ... 220/420
  ... 240/420
  ... 260/420
  ... 280/420
  ... 300/420
  ... 320/420
  ... 340/420
  ... 360/420
  ... 380/420
  ... 400/420
  ... 420/420
MiMo: 本轮完成 420 条
MiMo: 合计有效 420/420

模型 MiMo 标注完成，有效条数: 420/420
→ 换模型请修改配置 cell 的 MODEL_KEY，然后重跑「执行标注」+「导出结果 CSV」两个 cell


## 导出结果 CSV

结果写入**新文件**，按提示词模式分开存放，列结构与源文件一致（`类型`、`评论` + 各模型两列）：

| `PROMPT_MODE` | 输出文件 |
| --- | --- |
| `engineered`（提示词工程） | `Data/LLM/LLM_prompt_predictions.csv` |
| `zero_shot`（零样本） | `Data/LLM/LLM_predictions.csv` |

In [8]:
# 组装并合并写入（只覆盖当前模型的列）
_other = "zero_shot" if PROMPT_MODE == "engineered" else "engineered"
print(f"提示词模式: {PROMPT_MODE}")
print(f"本次写入:   {OUT_PATH}")
print(f"另一文件:   {OUT_FILES[_other]}（本次不写入，内容原样保留）\n")

# ---- 1) 取读数：优先在已有文件上增量更新，没有则按源文件结构新建 ----
if OUT_PATH.exists():
    df_out = pd.read_csv(OUT_PATH)
    _same = (len(df_out) == len(df_in)
             and df_out["评论"].astype(str).equals(df_in["评论"].astype(str)))
    if not _same:
        print("[警告] 已有结果文件与源数据不一致（行数或评论内容不匹配），改为以源文件重建")
        df_out = df_in[["类型", "评论"]].copy()
    else:
        print(f"读取已有结果: {OUT_PATH.name}")
else:
    df_out = df_in[["类型", "评论"]].copy()
    print(f"未发现 {OUT_PATH.name}，以源文件结构新建")

# ---- 2) 补齐所有模型列（缺失补空，保证列结构完整）----
for _m in MODELS:
    for _p in ("负面性", "侮辱性"):
        _c = f"{_p}{_m}"
        if _c not in df_out.columns:
            df_out[_c] = pd.NA

# ---- 3) 只填当前模型的列（其余模型列原样保留）----
res = globals().get("results", {}).get(MODEL_KEY, {})
if not res:
    print(f"\n[跳过写入] 本次没有 {MODEL_KEY} 的结果（API 未配置，或未运行「执行标注」cell）。")
    print(f"           为避免清空已有结果，{OUT_PATH.name} 内容未改动。")
else:
    _neg_col, _ins_col = f"负面性{MODEL_KEY}", f"侮辱性{MODEL_KEY}"
    _new_neg = pd.Series([res.get(i, {}).get("负面性") for i in range(len(df_in))],
                         index=df_out.index)
    _new_ins = pd.Series([res.get(i, {}).get("侮辱性") for i in range(len(df_in))],
                         index=df_out.index)
    # 新值非空才覆盖，否则保留旧值（防「部分调用失败」把已跑好的结果清掉）
    df_out[_neg_col] = _new_neg.combine_first(df_out[_neg_col])
    df_out[_ins_col] = _new_ins.combine_first(df_out[_ins_col])

# ---- 4) 按源文件列序输出 ----
df_out = df_out[["类型", "评论"] + [c for c in LABEL_COLS]]
if res:
    df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"已写入: {OUT_PATH}  形状 {df_out.shape}")

# ---- 5) 各模型完成进度（一眼看出还差哪个模型没跑）----
print(f"\n{OUT_PATH.name} 各模型完成情况:")
for _m in MODELS:
    _n = df_out[f"负面性{_m}"].notna().sum()
    _mark = "  ← 本次写入" if (_m == MODEL_KEY and res) else ""
    print(f"  {_m:10} {_n}/{len(df_out)}{_mark}")

# ---- 6) 另一个模式的文件确认未被触碰 ----
_o = OUT_FILES[_other]
print(f"\n另一模式文件 {_o.name}: {'存在' if _o.exists() else '尚未生成'}（本次未写入）")

df_out.head()

提示词模式: zero_shot
本次写入:   ../Data/LLM/LLM_predictions.csv
另一文件:   ../Data/LLM/LLM_prompt_predictions.csv（本次不写入，内容原样保留）

读取已有结果: LLM_predictions.csv
已写入: ../Data/LLM/LLM_predictions.csv  形状 (420, 8)

LLM_predictions.csv 各模型完成情况:
  MiMo       420/420  ← 本次写入
  Deepseek   420/420
  Qwen       420/420

另一模式文件 LLM_prompt_predictions.csv: 存在（本次未写入）


/var/folders/yt/b96z1pj11z35bffvldp7796c0000gn/T/ipykernel_70891/878250845.py:40: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df_out[_neg_col] = _new_neg.combine_first(df_out[_neg_col])
/var/folders/yt/b96z1pj11z35bffvldp7796c0000gn/T/ipykernel_70891/878250845.py:41: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df_out[_ins_col] = _new_ins.combine_first(df_out[_ins_col])


,类型,评论,负面性MiMo,侮辱性MiMo,负面性Deepseek,侮辱性Deepseek,负面性Qwen,侮辱性Qwen
0,语气型差评,用料扎实到咬不动 牙医听了都连夜谢谢你们家🤣,1,0,1,0,1,0
1,语气型差评,菜里面竟然附赠了一条🐛蠕动的赠品，真是太贴心了😭,1,0,1,0,1,0
2,语气型差评,老板仁义啊！居然拿半个世纪前的古董来做菜！菜叶枯黄得像秋天的落叶，不得不分五期给五星好评了捏😋,1,0,1,0,1,0
3,语气型差评,点的微辣，结果辣得我全家连夜去医院挂肛肠科，真是一场酣畅淋漓的体验。,1,0,1,0,1,0
4,语气型差评,这菜的味道极其富有层次感，先是齁咸，接着是苦涩，最后是一股刷锅水味，厨师真是个调味鬼才,1,0,1,1,1,1
